In [24]:
import pandas as pd
import joblib
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split,GridSearchCV,KFold
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = pd.read_csv('cleaned_dataset.csv')
results = []

Split Data:

In [11]:
X = df.drop('Exam_Score',axis=1)
y = df['Exam_Score']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=10)

In [12]:
numerical_features = ['Hours_Studied', 'Attendance','Previous_Scores']
ordinal_features = [
  'Motivation_Level',
  'Access_to_Resources',
  'Teacher_Quality',
  'Family_Income',
  'Parental_Involvement',
  'Peer_Influence',
  'Distance_from_Home',
  'Parental_Education_Level'
]
nominal_features = [
  'Gender',
  'School_Type',
  'Extracurricular_Activities',
  'Internet_Access',
  'Learning_Disabilities'
]
preprocessor = ColumnTransformer(
  [
    ('num',StandardScaler(),numerical_features),
    ('ord',OrdinalEncoder(),ordinal_features),
    ('nom',OneHotEncoder(),nominal_features)
  ],
  remainder='passthrough'
)

Piplines:

Random Forest:

In [13]:
rf_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',RandomForestRegressor())
])
rf_pipeline.fit(X_train,y_train)
y_pred = rf_pipeline.predict(X_test)
results.append({
    'Model': 'Random Forest',
    'Stage': 'Before',
    'RMSE': (mean_squared_error(y_test,y_pred)**.5),
    'MAE': mean_absolute_error(y_test,y_pred),
    'R2': r2_score(y_test,y_pred)
})

In [18]:
cv = KFold(n_splits = 3,shuffle=True,random_state=10)
CV_rfr = GridSearchCV(estimator=rf_pipeline, param_grid={'model__n_estimators': [300,600,900],'model__min_samples_split': [20,100,500],'model__max_depth' : [3,5,7]}, cv= cv,scoring='r2')
CV_rfr.fit(X_train, y_train)
best_model = CV_rfr.best_estimator_
y_pred = best_model.predict(X_test)
results.append({
    'Model': 'Random Forest',
    'Stage': 'After',
    'RMSE': (mean_squared_error(y_test,y_pred)**.5),
    'MAE': mean_absolute_error(y_test,y_pred),
    'R2': r2_score(y_test,y_pred)
})

SVR:

In [19]:
svr_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',SVR())
])
svr_pipeline.fit(X_train,y_train)
y_pred = svr_pipeline.predict(X_test)
results.append({
    'Model': 'Support Vector Regression',
    'Stage': 'Before',
    'RMSE': (mean_squared_error(y_test,y_pred)**.5),
    'MAE': mean_absolute_error(y_test,y_pred),
    'R2': r2_score(y_test,y_pred)
})

In [16]:
cv = KFold(n_splits=3, shuffle=True, random_state=10)
svr_CV = GridSearchCV(estimator=svr_pipeline,param_grid={'model__C':[0.1,1,10,100],'model__kernel':['rbf','linear'],'model__epsilon': [0.01, 0.1, 0.5]},cv=cv,scoring='r2')
svr_CV.fit(X_train,y_train)
best_model = svr_CV.best_estimator_
y_pred = best_model.predict(X_test)
results.append({
    'Model': 'Support Vector Regression',
    'Stage': 'After',
    'RMSE': (mean_squared_error(y_test,y_pred)**.5),
    'MAE': mean_absolute_error(y_test,y_pred),
    'R2': r2_score(y_test,y_pred)
})


Linear Regression:

In [20]:
rl_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',LinearRegression())
])
rl_pipeline.fit(X_train,y_train)
y_pred = rl_pipeline.predict(X_test)
results.append({
    'Model': 'Linear Regression',
    'Stage': 'Before',
    'RMSE': (mean_squared_error(y_test,y_pred)**.5),
    'MAE': mean_absolute_error(y_test,y_pred),
    'R2': r2_score(y_test,y_pred)
})

In [21]:
cv = KFold(n_splits=3,shuffle=True,random_state=10)
lr_CV = GridSearchCV(estimator=rl_pipeline,param_grid={'model__fit_intercept':[True,False],'model__positive':[True,False]},cv=cv,scoring='r2')
lr_CV.fit(X_train,y_train)
best_model = lr_CV.best_estimator_
y_pred = best_model.predict(X_test)
results.append({
    'Model': 'Linear Regression',
    'Stage': 'After',
    'RMSE': (mean_squared_error(y_test,y_pred)**.5),
    'MAE': mean_absolute_error(y_test,y_pred),
    'R2': r2_score(y_test,y_pred)
})


Comparative Study:

In [22]:
results_df = pd.DataFrame(results)
results_df

,Model,Stage,RMSE,MAE,R2
0,Random Forest,Before,2.435682,1.174470,0.598553
1,Random Forest,Before,2.515815,1.357625,0.571703
2,Support Vector Regression,After,2.267316,0.857951,0.652134
3,Support Vector Regression,After,2.169062,0.503401,0.681630
4,Random Forest,After,2.516270,1.357604,0.571548
5,Support Vector Regression,Before,2.267316,0.857951,0.652134
6,Linear Regression,Before,2.413614,1.097421,0.605794
7,Linear Regression,After,2.413614,1.097421,0.605794


In [25]:
final_model = svr_CV.best_estimator_
joblib.dump(final_model, 'student_exam_model.joblib')

['student_exam_model.joblib']